# 03 - End-to-End Pipeline

This notebook runs the full Colab pipeline for encoder integration, feature extraction, CLIP target generation, and MLP training:
- environment setup
- Concerto package installation
- encoder execution on a real Area 5 room
- feature extraction with `scripts/extract_features.py`
- CLIP label embedding generation if needed
- MLP training with `src/train.py`
- checkpoint inspection


### 1. Mount Drive and Get the Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'dev/enc-mlp'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline


### 2. Install Project Dependencies

In [ ]:
!pip install -q -r requirements.txt


### 3. Install Concerto Package Mode

This notebook only installs Concerto when feature extraction is actually needed. If `features/s3dis_area5/*.npz` already exist it skips the brittle `spconv` path; otherwise it pins `torch 2.5.0 + cu124`, tests a few `spconv` wheels, and verifies imports in a fresh Python process.

In [ ]:
import glob
import os
import re
import subprocess
import sys

PINNED_TORCH = '2.5.0'
PINNED_TORCHVISION = '0.20.0'
PINNED_TORCHAUDIO = '2.5.0'
PINNED_TORCH_CUDA_TAG = '124'
CONCERTO_DIR = '/content/Concerto'
FEATURE_GLOB = 'features/s3dis_area5/*.npz'
DRIVE_FEATURE_GLOB = '/content/drive/MyDrive/DL_Project/features/s3dis_area5/*.npz'
FORCE_FEATURE_REEXTRACTION = False
os.environ['CONCERTO_DIR'] = CONCERTO_DIR

def run(
    cmd: str,
    check: bool = True,
    capture: bool = False,
    cwd: str | None = None,
) -> subprocess.CompletedProcess[str]:
    print(f'+ {cmd}')
    completed = subprocess.run(
        cmd,
        shell=True,
        check=False,
        text=True,
        capture_output=capture,
        cwd=cwd,
    )
    if capture and completed.stdout:
        print(completed.stdout.strip())
    if capture and completed.stderr:
        print(completed.stderr.strip())
    if check and completed.returncode != 0:
        raise RuntimeError(
            f'Command failed with exit code {completed.returncode}: {cmd}'
        )
    return completed

def detect_cuda_tag_from_nvcc() -> str | None:
    completed = run('nvcc --version', check=False, capture=True)
    if completed.returncode != 0:
        return None
    match = re.search(r'release\\s+(\\d+)\\.(\\d+)', completed.stdout + completed.stderr)
    if match is None:
        return None
    return f"{match.group(1)}{match.group(2)}"

existing_feature_files = sorted(set(glob.glob(FEATURE_GLOB) + glob.glob(DRIVE_FEATURE_GLOB)))
REQUIRE_CONCERTO = FORCE_FEATURE_REEXTRACTION or not existing_feature_files

print('existing feature files:', len(existing_feature_files))
print('force re-extraction  :', FORCE_FEATURE_REEXTRACTION)
print('require Concerto     :', REQUIRE_CONCERTO)

if REQUIRE_CONCERTO:
    run('python --version')
    run('pip uninstall -y torch torchvision torchaudio spconv spconv-cu118 spconv-cu120 spconv-cu121 spconv-cu124 spconv-cu125 spconv-cu126 spconv-cu128 cumm cumm-cu118 cumm-cu120 cumm-cu121 cumm-cu124 cumm-cu125 cumm-cu126 cumm-cu128 torch-scatter || true')
    run(
        f'pip install --no-cache-dir torch=={PINNED_TORCH} torchvision=={PINNED_TORCHVISION} torchaudio=={PINNED_TORCHAUDIO} --index-url https://download.pytorch.org/whl/cu{PINNED_TORCH_CUDA_TAG}'
    )

    import torch

    if torch.version.cuda is None:
        raise RuntimeError(
            'CUDA-enabled PyTorch was not installed. Restart the runtime on a T4 GPU and rerun from the top.'
        )

    torch_version = torch.__version__.split('+')[0]
    torch_cuda_tag = torch.version.cuda.replace('.', '')
    nvcc_cuda_tag = detect_cuda_tag_from_nvcc()

    print('torch version :', torch.__version__)
    print('torch cuda tag:', torch_cuda_tag)
    print('nvcc cuda tag :', nvcc_cuda_tag)

    if torch_version != PINNED_TORCH or torch_cuda_tag != PINNED_TORCH_CUDA_TAG:
        raise RuntimeError(
            f'Expected torch {PINNED_TORCH}+cu{PINNED_TORCH_CUDA_TAG}, got {torch.__version__}.'
        )

    if not os.path.exists(CONCERTO_DIR):
        run(f'git clone https://github.com/Pointcept/Concerto.git {CONCERTO_DIR}')
    else:
        run('git pull', cwd=CONCERTO_DIR)

    spconv_candidates = []
    for candidate in [nvcc_cuda_tag, torch_cuda_tag, '126', '125', '124', '121', '120', '118']:
        if candidate and candidate not in spconv_candidates:
            spconv_candidates.append(candidate)

    spconv_ok = False
    last_spconv_error = None
    for spconv_cuda_tag in spconv_candidates:
        run('pip uninstall -y spconv spconv-cu118 spconv-cu120 spconv-cu121 spconv-cu124 spconv-cu125 spconv-cu126 spconv-cu128 cumm cumm-cu118 cumm-cu120 cumm-cu121 cumm-cu124 cumm-cu125 cumm-cu126 cumm-cu128 || true')
        install_result = run(
            f'pip install --no-cache-dir spconv-cu{spconv_cuda_tag}',
            check=False,
            capture=True,
        )
        if install_result.returncode != 0:
            last_spconv_error = install_result.stderr or install_result.stdout
            print(f'spconv-cu{spconv_cuda_tag} install failed, trying next candidate...')
            continue

        import_result = run(
            'python -c "import spconv.pytorch as spconv; print(\'spconv import ok:\', spconv.__file__)"',
            check=False,
            capture=True,
        )
        if import_result.returncode == 0:
            spconv_ok = True
            print(f'Selected spconv-cu{spconv_cuda_tag}.')
            break

        last_spconv_error = import_result.stderr or import_result.stdout
        print(f'spconv-cu{spconv_cuda_tag} import failed, trying next candidate...')

    if not spconv_ok:
        raise RuntimeError(
            'Could not import spconv in a fresh Python process. '
            'Factory-reset the Colab runtime and rerun this cell from the top. '
            f'Last error:\\n{last_spconv_error}'
        )

    run(
        f'pip install torch-scatter -f https://data.pyg.org/whl/torch-{PINNED_TORCH}+cu{PINNED_TORCH_CUDA_TAG}.html'
    )
    run(
        'python -c "import torch_scatter; print(\'torch_scatter import ok:\', torch_scatter.__file__)"'
    )

    run('pip install huggingface_hub timm')
    run('pip install -e .', cwd=CONCERTO_DIR)
    if CONCERTO_DIR not in sys.path:
        sys.path.insert(0, CONCERTO_DIR)
    import concerto
    print('concerto import ok:', concerto.__file__)
else:
    print('Skipping Concerto installation because feature files already exist.')


### 4. Symlink Shared Drive Folders

In [ ]:
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features
!ln -sf /content/drive/MyDrive/DL_Project/pretrained ./pretrained


### 5. Verify Inputs

In [ ]:
from pathlib import Path
from src.dataset import S3DISDataset

if 'REQUIRE_CONCERTO' not in globals():
    raise RuntimeError('Run Section 3 first so the notebook can decide whether to install Concerto or reuse existing features.')

processed_dir = Path('data/s3dis_processed')
features_dir = Path('features/s3dis_area5')
clip_emb_path = processed_dir / 'label_to_clip_embeddings.npy'
checkpoint_path = Path('pretrained/concerto_small.pth')
ckpt_dir = Path('checkpoints/mlp_s3dis')
feature_files = sorted(features_dir.glob('*.npz'))

print('processed dir exists :', processed_dir.exists(), processed_dir)
print('Area_5 exists        :', (processed_dir / 'Area_5').exists(), processed_dir / 'Area_5')
print('features dir exists  :', features_dir.exists(), features_dir)
print('existing feature files:', len(feature_files))
print('CLIP embeddings file :', clip_emb_path.exists(), clip_emb_path)
print('Concerto checkpoint  :', checkpoint_path.exists(), checkpoint_path)
print('checkpoint dir exists:', ckpt_dir.exists(), ckpt_dir)

dataset = S3DISDataset(root=str(processed_dir), areas=[5])
print('Area 5 rooms:', len(dataset))
print('sample rooms:', [dataset.rooms[i].name for i in range(min(5, len(dataset.rooms)))])

assert len(dataset) > 0, 'No processed Area_5 rooms found in data/s3dis_processed.'
if REQUIRE_CONCERTO:
    assert checkpoint_path.exists(), 'Missing pretrained/concerto_small.pth.'
else:
    print('Checkpoint not required because existing feature files will be reused.')


### 6. Run the Encoder on a Real Room

This runs the real Concerto encoder on one Area 5 room before the full extraction.

In [ ]:
if REQUIRE_CONCERTO:
    import numpy as np
    import torch

    from src.encoder import ConcertoEncoder

    sample = dataset[0]
    coord = sample['coord'].astype(np.float32)
    color = sample['color'].astype(np.float32)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    encoder = ConcertoEncoder(
        checkpoint_path='pretrained/concerto_small.pth',
        device=device,
    )
    encoder_input = {
        'coord': coord,
        'color': color,
    }
    with torch.no_grad():
        room_features = encoder(encoder_input)

    print('room          :', sample['room'])
    print('input shape   :', coord.shape)
    print('feature shape :', tuple(room_features.shape))
    print('all finite    :', torch.isfinite(room_features).all().item())
else:
    print('Skipping encoder smoke test because existing feature files will be reused.')


### 7. Extract Concerto Features for Area 5

This writes `.npz` files to `features/s3dis_area5/`. Re-run with `--overwrite` only if you want to rebuild them.

In [ ]:
if REQUIRE_CONCERTO:
    overwrite_flag = '--overwrite' if FORCE_FEATURE_REEXTRACTION else ''
    command = f'python scripts/extract_features.py --checkpoint pretrained/concerto_small.pth {overwrite_flag}'.strip()
    run(command)
else:
    print('Skipping feature extraction because existing .npz files will be reused.')


In [ ]:
from pathlib import Path
import numpy as np

feature_files = sorted(Path('features/s3dis_area5').glob('*.npz'))
print('num feature files:', len(feature_files))
print('sample files     :', [path.name for path in feature_files[:5]])

assert feature_files, 'No .npz feature files were produced.'

sample = np.load(feature_files[0])
print('sample keys:', sample.files)
for key in sample.files:
    print(key, sample[key].shape, sample[key].dtype)


### 8. Generate CLIP Label Embeddings if Missing

In [ ]:
from pathlib import Path

clip_emb_path = Path('data/s3dis_processed/label_to_clip_embeddings.npy')
if clip_emb_path.exists():
    print('CLIP label embeddings already exist:', clip_emb_path)
else:
    !python src/clip_utils.py


### 9. Inspect Training Config

In [ ]:
from pathlib import Path
import yaml

config_path = Path('configs/train_mlp_s3dis.yaml')
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
cfg


### 10. Train the Translation Head

In [ ]:
!python src/train.py --config configs/train_mlp_s3dis.yaml


### 11. Inspect Training Outputs

In [ ]:
from pathlib import Path
import json

ckpt_dir = Path('checkpoints/mlp_s3dis')
print('checkpoint dir exists:', ckpt_dir.exists())
for path in sorted(ckpt_dir.glob('*')):
    print(path.name)

history_path = ckpt_dir / 'history.json'
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
    print('\nLast history rows:')
    for row in history[-5:]:
        print(row)
else:
    print('\nNo history.json found yet.')
